# 第 1 周末练习 —— API 设计顾问（OpenRouter）

## 练习目标（理念）

为了熟悉 **OpenAI 兼容 API**（这里走 **OpenRouter**），构建一个能审阅 API 设计并给出解释的小工具：

- **输入**：一份有问题的 REST API 设计（写在 `question` 里）
- **输出**：从 RESTful、安全、命名、错误处理等角度给反馈
- **模型**：`openai/gpt-4o-mini` 与 `meta-llama/llama-3.2-3b-instruct`（都经 OpenRouter）

## 和本课的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions | `openrouter.chat.completions.create(...)` |
| `messages`（system / user） | system 定「API 架构师」角色，user 放设计稿 |
| 环境变量 | `.env` 里的 `OPENROUTER_API_KEY` + `load_dotenv` |

## 怎么跑

1. `.env` 中配置 `OPENROUTER_API_KEY`
2. 从上到下运行；可改 `question` 换一份 API 设计再问
3. 注意：GPT 流式单元格若只有标题打印、未调用 API，属原作者未写完；Llama 格为完整非流式调用


In [ ]:
# ========== 导入：环境、OpenAI SDK、dotenv ==========

# 导入标准库 os：读环境变量（Environment Variables）
import os
# 从 openai 导入 OpenAI：这里会指向 OpenRouter 的兼容端点
from openai import OpenAI
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进进程环境
from dotenv import load_dotenv


In [ ]:
# ========== 常量：模型 id 与 OpenRouter 根地址 ==========

# OpenRouter 上的 GPT 小模型路由名（带 openai/ 前缀，按 OpenRouter 目录填写）
MODEL_GPT = "openai/gpt-4o-mini"
# OpenRouter 上的 Llama 3.2 指令模型路由名
MODEL_LLAMA = "meta-llama/llama-3.2-3b-instruct"
# OpenRouter OpenAI 兼容 API 的 base_url（不要改成别的网关，除非你有意切换）
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"


In [ ]:
# ========== 环境初始化：读密钥并创建 OpenRouter 客户端 ==========

# override=True：.env 里的值覆盖已有环境变量（按原代码保留该参数）
load_dotenv(override=True)
# 取出 OpenRouter API Key；未配置时后面请求会失败
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
# 一个客户端同时服务 GPT 与 Llama（靠 model 参数区分）
openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)

# 给人看的就绪提示（字符串原样保留）
print("✅ Setup complete!")
print(f"openrouter defined: {openrouter is not None}")


In [ ]:
# ========== 问题（user prompt）：一份待评审的 API 设计 ==========

# 改这个多行字符串即可换题；正文是发给模型的 prompt，保持英文原样
question = """
Act as an API Design Consultant. Review this API design and provide feedback:

API: User Service
Endpoint: POST /api/v1/saveuser
Request Body: {
    "name": "John",
    "email": "john@email.com",
    "pass": "123456",
    "type": "admin"
}
Response: {
    "success": true,
    "message": "User saved"
}

Please analyze:
1. RESTful best practices
2. Security issues
3. Naming conventions
4. Error handling suggestions
5. Improvements
"""

# 确认 question 已载入内存
print("✅ Question loaded")


In [ ]:
# ========== GPT-4o-mini 流式回答（标题区）==========
# 说明：原作者此处只打印了横幅，尚未写出 chat.completions.create；逻辑保持原样不补全

# 让 gpt-4o-mini 接听，带流媒体（streaming）——下面三行是输出分隔标题
print("="*70)
print("🔍 API DESIGN CONSULTANT - GPT-4o-mini (STREAMING)")
print("="*70)


In [ ]:
# ========== Llama 3.2：经 OpenRouter 非流式回答 ==========

# 打印分隔标题
print("\n" + "="*70)
print("🦙 API DESIGN CONSULTANT - LLAMA 3.2")
print("="*70)

# 调用 Chat Completions：model 用 MODEL_LLAMA；stream=False 等整段完成
response = openrouter.chat.completions.create(
    model=MODEL_LLAMA,
    messages=[
        # system：API 架构师角色；content 保留英文原样
        {"role": "system", "content": "You are an API architect specializing in RESTful design and security."},
        # user：上面加载的 API 设计评审问题
        {"role": "user", "content": question}
    ],
    stream=False
)

# 取出助手完整回复文本
llama_response = response.choices[0].message.content
# 打印回答与收尾分隔线
print(llama_response)
print("="*70)
